# IAQP Reproduction 2: Evaluation and Generalization

This notebook runs the evaluation protocol behind the paper results:

- in-domain checkpoint shootouts on CAGRA and IVF
- cross-backend generalization
- cross-dataset generalization with shared PCA
- Wasserstein / OOD analysis

All outputs are written under `notebooks/outputs/`.

In [ ]:
from pathlib import Path

REPO_ROOT = Path('..').resolve()
OUTPUT_ROOT = REPO_ROOT / 'notebooks' / 'outputs'
EVAL_ROOT = OUTPUT_ROOT / 'eval_results'
WASSERSTEIN_ROOT = OUTPUT_ROOT / 'wasserstein'

EVAL_ROOT, WASSERSTEIN_ROOT

## One-shot paper evaluation

This is the cleanest end-to-end command. It runs the comprehensive checkpoint shootouts plus the cross-dataset transfer suite and skips experiments whose JSON outputs already exist.

In [ ]:
%%bash
cd ..
python notebooks/regenerate_all_results.py


## Run a single checkpoint-shootout experiment

Use this when you want to rerun only one dataset/backend pair.

In [ ]:
%%bash
cd ..

CUDA_VISIBLE_DEVICES=0 python scripts/ckpt_shootout_comprehensive.py \
  --dataset laion \
  --size 10m \
  --train_backend cagra \
  --full_dataset \
  --at_k 10,50,100


## Cross-dataset transfer with shared PCA

The paper reports LAION↔DataComp transfer under a shared PCA rotation. Run both directions for each backend.

In [ ]:
%%bash
cd ..

CUDA_VISIBLE_DEVICES=0 python scripts/dataset_shootout.py \
  --dataset_trained laion \
  --dataset_test datacomp \
  --train_backend cagra \
  --size_trained 10m \
  --size_test 8.2m \
  --pca_regime ckpt \
  --shared_pca \
  --full_dataset

CUDA_VISIBLE_DEVICES=0 python scripts/dataset_shootout.py \
  --dataset_trained datacomp \
  --dataset_test laion \
  --train_backend ivf \
  --size_trained 8.2m \
  --size_test 10m \
  --pca_regime ckpt \
  --shared_pca \
  --full_dataset


## OOD / Wasserstein analysis

Section 5 of the paper measures Gaussian 2-Wasserstein distance between image-bank and query distributions before and after IAQP.

In [ ]:
%%bash
cd ..
python scripts/compute_wasserstein_distances.py \
  --output notebooks/outputs/wasserstein/wasserstein_distances.json


## Inspect generated evaluation files

These JSON files are the main machine-readable outputs consumed by the paper analysis scripts.

In [ ]:
from pathlib import Path

eval_files = sorted((Path('..') / 'notebooks' / 'outputs' / 'eval_results').glob('*.json'))
for path in eval_files:
    print(path.name)
